In [ ]:
# ============================================
# FEDERATED SPLIT (MODERATE NON-IID - STABLE)
# ============================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# CONFIG
# =========================
TRAIN_DATASET = "processed_train.csv"
TEST_DATASET = "processed_test.csv"
OUTPUT_DIR = "fed-data-100426"

NUM_CLIENTS = 7
LOCAL_TEST_RATIO = 0.20
DIRICHLET_ALPHA = 1.0   #

RANDOM_STATE = 42
LABEL_COL = "BinaryLabel"

MIN_SAMPLES_PER_CLASS = 50  # 🔥 prevents extreme imbalance

np.random.seed(RANDOM_STATE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# LOAD DATA
# =========================
print("Loading datasets...")

df_train = pd.read_csv(TRAIN_DATASET)
df_test = pd.read_csv(TEST_DATASET)

print(f"Train rows: {len(df_train)}")
print(f"Test rows : {len(df_test)}")

# =========================
# SAVE GLOBAL TEST SET
# =========================
print("\nSaving global test set...")

test_dir = os.path.join(OUTPUT_DIR, "global_test")
os.makedirs(test_dir, exist_ok=True)

X_test = df_test.drop(columns=[LABEL_COL]).to_numpy(dtype="float32")
y_test = df_test[LABEL_COL].to_numpy(dtype="int64")

np.save(os.path.join(test_dir, "X_test.npy"), X_test)
np.save(os.path.join(test_dir, "y_test.npy"), y_test)

print("✅ Global test set saved")

# =========================
# CREATE MODERATE NON-IID SPLIT
# =========================
print("\nCreating moderate non-IID client splits...")

clients = {i: [] for i in range(NUM_CLIENTS)}

for cls in sorted(df_train[LABEL_COL].unique()):

    cls_df = df_train[df_train[LABEL_COL] == cls]
    cls_indices = cls_df.index.to_numpy().copy()

    np.random.shuffle(cls_indices)

    # Dirichlet proportions
    proportions = np.random.dirichlet(
        alpha=[DIRICHLET_ALPHA] * NUM_CLIENTS
    )

    counts = (proportions * len(cls_indices)).astype(int)

    # Ensure minimum samples per client
    counts[counts < MIN_SAMPLES_PER_CLASS] = MIN_SAMPLES_PER_CLASS

    # Normalize again
    counts = (counts / counts.sum() * len(cls_indices)).astype(int)

    # Fix rounding
    diff = len(cls_indices) - np.sum(counts)
    counts[0] += diff

    # Split indices
    split_points = np.cumsum(counts)[:-1]
    splits = np.split(cls_indices, split_points)

    for cid in range(NUM_CLIENTS):
        clients[cid].append(df_train.loc[splits[cid]])

# =========================
# PROCESS CLIENTS
# =========================
print("\nProcessing clients...")

for cid in range(NUM_CLIENTS):

    client_df = pd.concat(clients[cid]).sample(frac=1, random_state=RANDOM_STATE)

    print(f"\nClient {cid} total samples: {len(client_df)}")
    print("Class distribution:")
    print(client_df[LABEL_COL].value_counts())

    # ---- LOCAL TRAIN / TEST SPLIT ----
    train_df, test_df_local = train_test_split(
        client_df,
        test_size=LOCAL_TEST_RATIO,
        stratify=client_df[LABEL_COL],
        random_state=RANDOM_STATE
    )

    # ---- CREATE DIR ----
    client_dir = os.path.join(OUTPUT_DIR, f"client_{cid}")
    os.makedirs(client_dir, exist_ok=True)

    # ---- TRAIN ----
    X_train = train_df.drop(columns=[LABEL_COL]).to_numpy(dtype="float32")
    y_train = train_df[LABEL_COL].to_numpy(dtype="int64")

    np.save(os.path.join(client_dir, "X_train.npy"), X_train)
    np.save(os.path.join(client_dir, "y_train.npy"), y_train)

    # ---- TEST ----
    X_test_local = test_df_local.drop(columns=[LABEL_COL]).to_numpy(dtype="float32")
    y_test_local = test_df_local[LABEL_COL].to_numpy(dtype="int64")

    np.save(os.path.join(client_dir, "X_test.npy"), X_test_local)
    np.save(os.path.join(client_dir, "y_test.npy"), y_test_local)

    print(f"Train: {len(train_df)}, Test: {len(test_df_local)}")

print("\n✅ Moderate non-IID federated dataset created!")

In [1]:
# New 11 April 2026

In [2]:
# ============================================
# FEDERATED SPLIT (OPTIMAL MODERATE NON-IID)
# BEST FOR ACCURACY + F1 (BINARY)
# ============================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# CONFIG
# =========================
TRAIN_DATASET = "processed_train.csv"
TEST_DATASET = "processed_test.csv"
OUTPUT_DIR = "fed-data-110426"

NUM_CLIENTS = 7
LOCAL_TEST_RATIO = 0.20
RANDOM_STATE = 42
LABEL_COL = "BinaryLabel"

# 🔥 CONTROL NON-IID STRENGTH
BIAS_RANGE = 0.15   # ±15% variation (sweet spot)

np.random.seed(RANDOM_STATE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# LOAD DATA
# =========================
print("Loading datasets...")

df_train = pd.read_csv(TRAIN_DATASET)
df_test = pd.read_csv(TEST_DATASET)

print(f"Train rows: {len(df_train)}")
print(f"Test rows : {len(df_test)}")

# =========================
# SAVE GLOBAL TEST SET
# =========================
print("\nSaving global test set...")

test_dir = os.path.join(OUTPUT_DIR, "global_test")
os.makedirs(test_dir, exist_ok=True)

X_test = df_test.drop(columns=[LABEL_COL]).to_numpy(dtype="float32")
y_test = df_test[LABEL_COL].to_numpy(dtype="int64")

np.save(os.path.join(test_dir, "X_test.npy"), X_test)
np.save(os.path.join(test_dir, "y_test.npy"), y_test)

print("✅ Global test set saved")

# =========================
# GLOBAL CLASS DISTRIBUTION
# =========================
class_counts = df_train[LABEL_COL].value_counts(normalize=True)
p_benign = class_counts[0]
p_attack = class_counts[1]

print("\nGlobal distribution:")
print(class_counts)

# =========================
# CREATE CONTROLLED NON-IID CLIENTS
# =========================
print("\nCreating optimized non-IID clients...")

df_benign = df_train[df_train[LABEL_COL] == 0].sample(frac=1, random_state=RANDOM_STATE)
df_attack = df_train[df_train[LABEL_COL] == 1].sample(frac=1, random_state=RANDOM_STATE)

benign_ptr = 0
attack_ptr = 0

clients = []

total_samples = len(df_train)
samples_per_client = total_samples // NUM_CLIENTS

for cid in range(NUM_CLIENTS):

    # 🔥 Slight bias around global distribution
    bias = np.random.uniform(-BIAS_RANGE, BIAS_RANGE)

    p_client_attack = np.clip(p_attack + bias, 0.2, 0.8)
    p_client_benign = 1 - p_client_attack

    n_attack = int(samples_per_client * p_client_attack)
    n_benign = int(samples_per_client * p_client_benign)

    # Sample safely
    attack_samples = df_attack.iloc[attack_ptr:attack_ptr + n_attack]
    benign_samples = df_benign.iloc[benign_ptr:benign_ptr + n_benign]

    attack_ptr += n_attack
    benign_ptr += n_benign

    client_df = pd.concat([attack_samples, benign_samples]).sample(frac=1, random_state=RANDOM_STATE)

    clients.append(client_df)

# =========================
# PROCESS CLIENTS
# =========================
print("\nProcessing clients...")

for cid, client_df in enumerate(clients):

    print(f"\nClient {cid} samples: {len(client_df)}")
    print(client_df[LABEL_COL].value_counts(normalize=True))

    # ---- LOCAL TRAIN / TEST SPLIT ----
    train_df, test_df_local = train_test_split(
        client_df,
        test_size=LOCAL_TEST_RATIO,
        stratify=client_df[LABEL_COL],
        random_state=RANDOM_STATE
    )

    client_dir = os.path.join(OUTPUT_DIR, f"client_{cid}")
    os.makedirs(client_dir, exist_ok=True)

    # ---- TRAIN ----
    X_train = train_df.drop(columns=[LABEL_COL]).to_numpy(dtype="float32")
    y_train = train_df[LABEL_COL].to_numpy(dtype="int64")

    np.save(os.path.join(client_dir, "X_train.npy"), X_train)
    np.save(os.path.join(client_dir, "y_train.npy"), y_train)

    # ---- TEST ----
    X_test_local = test_df_local.drop(columns=[LABEL_COL]).to_numpy(dtype="float32")
    y_test_local = test_df_local[LABEL_COL].to_numpy(dtype="int64")

    np.save(os.path.join(client_dir, "X_test.npy"), X_test_local)
    np.save(os.path.join(client_dir, "y_test.npy"), y_test_local)

    print(f"Train: {len(train_df)}, Test: {len(test_df_local)}")

print("\n✅ Optimized non-IID dataset created (BEST for accuracy + F1)")

Loading datasets...
Train rows: 880000
Test rows : 220000

Saving global test set...
✅ Global test set saved

Global distribution:
BinaryLabel
0    0.78387
1    0.21613
Name: proportion, dtype: float64

Creating optimized non-IID clients...

Processing clients...

Client 0 samples: 125713
BinaryLabel
0    0.800005
1    0.199995
Name: proportion, dtype: float64
Train: 100570, Test: 25143

Client 1 samples: 125713
BinaryLabel
0    0.64866
1    0.35134
Name: proportion, dtype: float64
Train: 100570, Test: 25143

Client 2 samples: 125713
BinaryLabel
0    0.714278
1    0.285722
Name: proportion, dtype: float64
Train: 100570, Test: 25143

Client 3 samples: 125713
BinaryLabel
0    0.754274
1    0.245726
Name: proportion, dtype: float64
Train: 100570, Test: 25143

Client 4 samples: 125713
BinaryLabel
0    0.800005
1    0.199995
Name: proportion, dtype: float64
Train: 100570, Test: 25143

Client 5 samples: 125713
BinaryLabel
0    0.800005
1    0.199995
Name: proportion, dtype: float64
Train: 10

In [7]:
# ============================================
# FEDERATED SPLIT (FINAL SAFE VERSION)
# NO SKLEARN • NO RECURSION • MAX ACCURACY
# ============================================

import os
import numpy as np
import pandas as pd

# =========================
# CONFIG
# =========================
TRAIN_DATASET = "processed_train.csv"
TEST_DATASET = "processed_test.csv"
OUTPUT_DIR = "fed-data-final"

NUM_CLIENTS = 7
LOCAL_TEST_RATIO = 0.20
RANDOM_STATE = 42
LABEL_COL = "BinaryLabel"

BIAS_RANGE = 0.03   # 🔥 small variation (best)

np.random.seed(RANDOM_STATE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# LOAD DATA
# =========================
print("Loading datasets...")

df_train = pd.read_csv(TRAIN_DATASET)
df_test = pd.read_csv(TEST_DATASET)

print(f"Train rows: {len(df_train)}")
print(f"Test rows : {len(df_test)}")

# =========================
# CONVERT TO NUMPY (IMPORTANT 🔥)
# =========================
X_all = df_train.drop(columns=[LABEL_COL]).values.astype("float32")
y_all = df_train[LABEL_COL].values.astype("int64")

X_test_global = df_test.drop(columns=[LABEL_COL]).values.astype("float32")
y_test_global = df_test[LABEL_COL].values.astype("int64")

# =========================
# SAVE GLOBAL TEST (NO LEAKAGE)
# =========================
print("\nSaving global test set...")

test_dir = os.path.join(OUTPUT_DIR, "global_test")
os.makedirs(test_dir, exist_ok=True)

np.save(os.path.join(test_dir, "X_test.npy"), X_test_global)
np.save(os.path.join(test_dir, "y_test.npy"), y_test_global)

print("✅ Global test set saved")

# =========================
# SHUFFLE DATA
# =========================
indices = np.arange(len(X_all))
np.random.shuffle(indices)

X_all = X_all[indices]
y_all = y_all[indices]

# =========================
# GLOBAL CLASS DISTRIBUTION
# =========================
p_attack = np.mean(y_all)
print(f"\nGlobal attack ratio: {p_attack:.4f}")

# =========================
# SPLIT INTO CLIENTS (IID BASE)
# =========================
client_indices = np.array_split(np.arange(len(X_all)), NUM_CLIENTS)

print("\nCreating near-IID clients...")

# =========================
# PROCESS CLIENTS
# =========================
for cid, idx in enumerate(client_indices):

    X_client = X_all[idx]
    y_client = y_all[idx]

    # 🔥 small variation
    bias = np.random.uniform(-BIAS_RANGE, BIAS_RANGE)
    target_ratio = np.clip(p_attack + bias, 0.4, 0.6)

    # Separate classes
    attack_idx = np.where(y_client == 1)[0]
    benign_idx = np.where(y_client == 0)[0]

    n_total = len(y_client)
    n_attack = int(n_total * target_ratio)

    # Select WITHOUT duplication
    attack_selected = attack_idx[:min(n_attack, len(attack_idx))]
    benign_selected = benign_idx[:(n_total - len(attack_selected))]

    selected_idx = np.concatenate([attack_selected, benign_selected])

    # Fill if needed
    if len(selected_idx) < n_total:
        remaining = np.setdiff1d(np.arange(n_total), selected_idx)
        needed = n_total - len(selected_idx)
        selected_idx = np.concatenate([selected_idx, remaining[:needed]])

    np.random.shuffle(selected_idx)

    X_client = X_client[selected_idx]
    y_client = y_client[selected_idx]

    print(f"\nClient {cid} samples: {len(y_client)}")
    unique, counts = np.unique(y_client, return_counts=True)
    print("Class distribution:", dict(zip(unique, counts)))

    # =========================
    # LOCAL TRAIN/TEST SPLIT
    # =========================
    split = int(len(X_client) * (1 - LOCAL_TEST_RATIO))

    X_train = X_client[:split]
    y_train = y_client[:split]

    X_test_local = X_client[split:]
    y_test_local = y_client[split:]

    # =========================
    # SAVE CLIENT DATA
    # =========================
    client_dir = os.path.join(OUTPUT_DIR, f"client_{cid}")
    os.makedirs(client_dir, exist_ok=True)

    np.save(os.path.join(client_dir, "X_train.npy"), X_train)
    np.save(os.path.join(client_dir, "y_train.npy"), y_train)

    np.save(os.path.join(client_dir, "X_test.npy"), X_test_local)
    np.save(os.path.join(client_dir, "y_test.npy"), y_test_local)

    print(f"Train: {len(y_train)}, Test: {len(y_test_local)}")

print("\n✅ Federated dataset created (NO ERROR, NO LEAKAGE)")

Loading datasets...
Train rows: 880000
Test rows : 220000

Saving global test set...
✅ Global test set saved

Global attack ratio: 0.2161

Creating near-IID clients...

Client 0 samples: 125715
Class distribution: {np.int64(0): np.int64(98409), np.int64(1): np.int64(27306)}
Train: 100572, Test: 25143

Client 1 samples: 125715
Class distribution: {np.int64(0): np.int64(98486), np.int64(1): np.int64(27229)}
Train: 100572, Test: 25143

Client 2 samples: 125714
Class distribution: {np.int64(0): np.int64(98476), np.int64(1): np.int64(27238)}
Train: 100571, Test: 25143

Client 3 samples: 125714
Class distribution: {np.int64(0): np.int64(98549), np.int64(1): np.int64(27165)}
Train: 100571, Test: 25143

Client 4 samples: 125714
Class distribution: {np.int64(0): np.int64(98625), np.int64(1): np.int64(27089)}
Train: 100571, Test: 25143

Client 5 samples: 125714
Class distribution: {np.int64(0): np.int64(98719), np.int64(1): np.int64(26995)}
Train: 100571, Test: 25143

Client 6 samples: 125714
Cl